# 02 — Transition / migration matrices & roll rates

**Plain English:** The headline of the whole monitor. A **transition matrix**
reads: *of the loans in this row's state this month, what share are in each
column's state next month?* Each row sums to 1. We build it at **monthly**
periodicity, for both delinquency buckets and IFRS 9 stages, and pull the key
**roll rates** (how fast loans deteriorate) and cure rates out of it.

**One-line terms**
- **Transition matrix** — period-over-period migration probabilities; the diagonal is "stayed put", above the diagonal is cure, below is deterioration.
- **Roll rate** — the share of a worse-bucket move, e.g. 30→60, 60→90+.
- **Cure rate** — the share that improves, e.g. 30→Current.

Periodicity = **one calendar month**.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")
print(f"{len(panel):,} loan-months loaded")

In [ ]:
# Pooled bucket transition matrix (all vintages) -------------------------
counts_b, probs_b = m.transition_matrix(panel, "bucket")
probs_b.round(4)

In [ ]:
# IFRS 9 stage transition matrix ----------------------------------------
counts_s, probs_s = m.transition_matrix(panel, "stage")
probs_s.round(4)

In [ ]:
# Headline roll rates / cure rates --------------------------------------
rr = m.roll_rates(probs_b)
rr["monthly_probability"] = rr["monthly_probability"].round(4)
rr

In [ ]:
# Heatmap of the bucket transition matrix (the headline visual) ----------
fig, ax = plt.subplots(figsize=(7, 4.2))
im = ax.imshow(probs_b.values, cmap="rocket_r" if "rocket_r" in plt.colormaps() else "magma_r",
               vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(probs_b.shape[1])); ax.set_xticklabels(probs_b.columns, rotation=30, ha="right")
ax.set_yticks(range(probs_b.shape[0])); ax.set_yticklabels(probs_b.index)
for i in range(probs_b.shape[0]):
    for j in range(probs_b.shape[1]):
        val = probs_b.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if val > 0.5 else "black", fontsize=9)
ax.set_xlabel("next month"); ax.set_ylabel("this month")
ax.set_title("Monthly delinquency-bucket transition matrix (all vintages)")
fig.colorbar(im, ax=ax, label="probability"); fig.tight_layout()
fig.savefig(m.OUT_CHARTS / "02_bucket_transition_heatmap.png", dpi=130)
print("saved heatmap"); plt.close(fig)

In [ ]:
# Save the clean tables --------------------------------------------------
probs_b.round(5).to_csv(m.OUT_TABLES / "02_bucket_transition_matrix.csv")
probs_s.round(5).to_csv(m.OUT_TABLES / "02_stage_transition_matrix.csv")
rr.to_csv(m.OUT_TABLES / "02_roll_rates.csv", index=False)
print("saved bucket + stage matrices and roll rates")